## Coding Exercise #0605

In [7]:
# Install Tweepy version 4 for access to the Twitter API v2.
!pip install tweepy==4

In [8]:
import tweepy
import re
import os
import pickle
import nltk
from nltk.corpus import stopwords
from tweepy import OAuthHandler

In [9]:
%%time
# You should download the NLTK data once.
# It can be a bit time consuming.
# nltk.download()
# We download just what we need.
nltk.download('punkt')
nltk.download('stopwords')

CPU times: user 29.8 ms, sys: 4.97 ms, total: 34.8 ms
Wall time: 36.1 ms


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 1. Fetch tweets
More information can be found [here](https://docs.tweepy.org/en/v4.0.0/index.html).

In [13]:
# PLEASE FILL IN WITH YOUR OWN API KEYS AND TOKENS!
bearer_token = 'YOUR_BEARER_TOKEN_HERE' # Paste your bearer token here

In [11]:
# Initialize a Client
client = tweepy.Client(bearer_token)

In [14]:
query = "Putin -is:retweet"  # The query string
n_tweets = 10  # The maximum number of tweets (default: 10, max: 100)

# Example tweets for demonstration since API token is not yet provided
tweets = [
    "Just had a fantastic meal at my favorite restaurant! #foodie",
    "Watching the sunset over the mountains. Absolutely breathtaking. #nature #beautiful",
    "Learning Python today! It's challenging but rewarding. #coding #programming",
    "New book release! Can't wait to dive in this weekend. #books #reading",
    "My cat just did the funniest thing. Wish I had caught it on camera! #cats #pets",
    "Productive day at work. Feeling accomplished! #work #motivation",
    "Planning my next vacation. Where should I go? Suggestions welcome! #travel #wanderlust",
    "Enjoying a quiet evening with a cup of tea. Simple pleasures. #relax #evening",
    "The new movie trailer looks amazing! Definitely going to see it. #movies #film",
    "Thinking about starting a new project. Any ideas for a beginner? #project #ideas"
]

print('\n\n'.join(tweets))

Just had a fantastic meal at my favorite restaurant! #foodie

Watching the sunset over the mountains. Absolutely breathtaking. #nature #beautiful

Learning Python today! It's challenging but rewarding. #coding #programming

New book release! Can't wait to dive in this weekend. #books #reading

My cat just did the funniest thing. Wish I had caught it on camera! #cats #pets

Productive day at work. Feeling accomplished! #work #motivation

Planning my next vacation. Where should I go? Suggestions welcome! #travel #wanderlust

Enjoying a quiet evening with a cup of tea. Simple pleasures. #relax #evening

The new movie trailer looks amazing! Definitely going to see it. #movies #film

Thinking about starting a new project. Any ideas for a beginner? #project #ideas


### 2. Pre-processing tweets

In [16]:
%%time
nltk.download('punkt_tab', quiet=True)
tweets_preprocessed = []
for tweet in tweets:
    tweet = tweet.lower()
    tweet = re.sub(r"^https://t.co/[a-zA-Z0-9]*\s", " ", tweet)
    tweet = re.sub(r"\s+https://t.co/[a-zA-Z0-9]*\s", " ", tweet)
    tweet = re.sub(r"\s+https://t.co/[a-zA-Z0-9]*$", " ", tweet)
    tweet = re.sub(r"that's", "that is", tweet)
    tweet = re.sub(r"there's", "there is", tweet)
    tweet = re.sub(r"what's", "what is", tweet)
    tweet = re.sub(r"where's", "where is", tweet)
    tweet = re.sub(r"it's", "it is", tweet)
    tweet = re.sub(r"who's", "who is", tweet)
    tweet = re.sub(r"i'm", "i am", tweet)
    tweet = re.sub(r"she's", "she is", tweet)
    tweet = re.sub(r"he's", "he is", tweet)
    tweet = re.sub(r"they're", "they are", tweet)
    tweet = re.sub(r"who're", "who are", tweet)
    tweet = re.sub(r"ain't", "am not",tweet)
    tweet = re.sub(r"wouldn't", "would not", tweet)
    tweet = re.sub(r"shouldn't", "should not", tweet)
    tweet = re.sub(r"can't", "can not",tweet)
    tweet = re.sub(r"couldn't", "could not", tweet)
    tweet = re.sub(r"won't", "will not", tweet)
    tweet = re.sub(r"\W", " ", tweet)
    tweet = re.sub(r"\d", " ", tweet)
    tweet = re.sub(r"\s+[a-z]\s+", " ", tweet)
    tweet = re.sub(r"\s+[a-z]$", " ", tweet)
    tweet = re.sub(r"^[a-z]\s+", " ", tweet)
    tweet = re.sub(r"\s+", " ", tweet)
    words = nltk.word_tokenize(tweet)
    words = [x for x in words if x not in stopwords.words('english')]
    tweet = ' '.join(words)
    tweets_preprocessed.append(tweet)

CPU times: user 420 ms, sys: 40.8 ms, total: 461 ms
Wall time: 1.2 s


In [17]:
# Print out the tweets.
for i, tweet in enumerate(tweets_preprocessed):
    print(f"{i}: {tweet}")

0: fantastic meal favorite restaurant foodie
1: watching sunset mountains absolutely breathtaking nature beautiful
2: learning python today challenging rewarding coding programming
3: new book release wait dive weekend books reading
4: cat funniest thing wish caught camera cats pets
5: productive day work feeling accomplished work motivation
6: planning next vacation go suggestions welcome travel wanderlust
7: enjoying quiet evening cup tea simple pleasures relax evening
8: new movie trailer looks amazing definitely going see movies film
9: thinking starting new project ideas beginner project ideas


#### Refactor the code and check for the same results

In [18]:
def preprocess_tweet(tweet):
    """Preprocess a tweet."""
    tweet = tweet.lower()
    tweet = re.sub(r"^https://t.co/[a-zA-Z0-9]*\s", " ", tweet)
    tweet = re.sub(r"\s+https://t.co/[a-zA-Z0-9]*\s", " ", tweet)
    tweet = re.sub(r"\s+https://t.co/[a-zA-Z0-9]*$", " ", tweet)
    tweet = re.sub(r"that's", "that is", tweet)
    tweet = re.sub(r"there's", "there is", tweet)
    tweet = re.sub(r"what's", "what is", tweet)
    tweet = re.sub(r"where's", "where is", tweet)
    tweet = re.sub(r"it's", "it is", tweet)
    tweet = re.sub(r"who's", "who is", tweet)
    tweet = re.sub(r"i'm", "i am", tweet)
    tweet = re.sub(r"she's", "she is", tweet)
    tweet = re.sub(r"he's", "he is", tweet)
    tweet = re.sub(r"they're", "they are", tweet)
    tweet = re.sub(r"who're", "who are", tweet)
    tweet = re.sub(r"ain't", "am not",tweet)
    tweet = re.sub(r"wouldn't", "would not", tweet)
    tweet = re.sub(r"shouldn't", "should not", tweet)
    tweet = re.sub(r"can't", "can not",tweet)
    tweet = re.sub(r"couldn't", "could not", tweet)
    tweet = re.sub(r"won't", "will not", tweet)
    tweet = re.sub(r"\W", " ", tweet)
    tweet = re.sub(r"\d", " ", tweet)
    tweet = re.sub(r"\s+[a-z]\s+", " ", tweet)
    tweet = re.sub(r"\s+[a-z]$", " ", tweet)
    tweet = re.sub(r"^[a-z]\s+", " ", tweet)
    tweet = re.sub(r"\s+", " ", tweet)
    words = nltk.word_tokenize(tweet)
    words = [x for x in words if x not in stopwords.words('english')]

    return ' '.join(words)

## Understanding Regular Expressions (the `re` module)

Regular expressions (often shortened to regex or regexp) are sequences of characters that define a search pattern. They are extremely powerful for matching, finding, and replacing text in strings.

Python's built-in `re` module provides operations for working with regular expressions.

### Basic Concepts:

*   **Patterns**: The regular expression itself, defining what you are looking for.
*   **Target String**: The text you are searching within.

### Key Functions in the `re` Module:

1.  **`re.search(pattern, string)`**: Scans through `string` looking for the first location where the `pattern` produces a match. Returns a match object if found, otherwise `None`.

    *Example: Checking if a string contains a number.*

    ```python
    import re
    text = "My phone number is 123-456-7890"
    match = re.search(r"\d+", text) # \d+ matches one or more digits
    if match:
        print(f"Found: {match.group(0)}")
    else:
        print("Not found")
    ```

2.  **`re.match(pattern, string)`**: Checks for a match only at the *beginning* of the `string`. Returns a match object if found, otherwise `None`.

    *Example: Checking if a string starts with "Hello".*

    ```python
    import re
    text1 = "Hello World"
    text2 = "Greetings, Hello World"

    match1 = re.match(r"Hello", text1)
    match2 = re.match(r"Hello", text2)

    print(f"Match 1: {match1.group(0) if match1 else 'None'}") # Output: Hello
    print(f"Match 2: {match2.group(0) if match2 else 'None'}") # Output: None
    ```

3.  **`re.findall(pattern, string)`**: Returns all non-overlapping matches of `pattern` in `string`, as a list of strings.

    *Example: Finding all numbers in a string.*

    ```python
    import re
    text = "I have 2 apples and 5 bananas."
    numbers = re.findall(r"\d+", text)
    print(f"Numbers found: {numbers}") # Output: ['2', '5']
    ```

4.  **`re.sub(pattern, repl, string)`**: Replaces all occurrences of `pattern` in `string` with `repl`.

    *Example: Replacing all digits with a hash symbol.*

    ```python
    import re
    text = "Item_123_abc_456"
    new_text = re.sub(r"\d", "#", text)
    print(f"Modified text: {new_text}") # Output: Item_###_abc_###
    ```

### Common Regular Expression Syntax:

*   **`.`**: Matches any single character (except newline).
*   **`*`**: Matches 0 or more occurrences of the preceding character/group.
*   **`+`**: Matches 1 or more occurrences of the preceding character/group.
*   **`?`**: Matches 0 or 1 occurrence of the preceding character/group.
*   **`{n}`**: Matches exactly `n` occurrences.
*   **`{n,m}`**: Matches `n` to `m` occurrences.
*   **`[ ]`**: Matches any one of the characters inside the brackets (e.g., `[aeiou]` for vowels).
*   **`[^ ]`**: Matches any character *not* inside the brackets.
*   **`^`**: Matches the beginning of the string.
*   **`$`**: Matches the end of the string.
*   **`\d`**: Matches any digit (0-9).
*   **`\D`**: Matches any non-digit character.
*   **`\w`**: Matches any word character (alphanumeric + underscore).
*   **`\W`**: Matches any non-word character.
*   **`\s`**: Matches any whitespace character.
*   **`\S`**: Matches any non-whitespace character.
*   **`|`**: Acts as an OR (e.g., `cat|dog` matches "cat" or "dog").
*   **`(` `)`**: Groups patterns together.

### Raw Strings (`r""`) and `re`

It's highly recommended to use raw strings (e.g., `r"\d+"`) for regular expression patterns in Python. This prevents backslashes from being interpreted as escape sequences by Python itself, ensuring they are passed directly to the `re` module for regex interpretation.

For example, `\n` normally means a newline character in Python. In a raw string `r'\n'`, it means a literal backslash followed by 'n', which could be part of a regex pattern. For regular expressions like `\d` (digit), using `r"\d"` is crucial so Python doesn't try to interpret `\d` as an escape sequence before the `re` module sees it.

In [19]:
%%time
preprocessed_tweets = map(preprocess_tweet, tweets)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.39 µs


In [20]:
def print_tweets(tweets):
    """Print tweets."""
    for i, tweet in enumerate(tweets):
        print(f"{i}: {tweet}")

In [21]:
print_tweets(preprocessed_tweets)

0: fantastic meal favorite restaurant foodie
1: watching sunset mountains absolutely breathtaking nature beautiful
2: learning python today challenging rewarding coding programming
3: new book release wait dive weekend books reading
4: cat funniest thing wish caught camera cats pets
5: productive day work feeling accomplished work motivation
6: planning next vacation go suggestions welcome travel wanderlust
7: enjoying quiet evening cup tea simple pleasures relax evening
8: new movie trailer looks amazing definitely going see movies film
9: thinking starting new project ideas beginner project ideas


### 3. Follow a Twitter user
More examples can be found [here](hhttps://dev.to/twitterdev/a-comprehensive-guide-for-using-the-twitter-api-v2-using-tweepy-in-python-15d9).

In [22]:
query = "from:elonmusk -is:retweet"  # Query
n_tweets = 10  # Maximum number of tweets (default: 10, max: 100)

response = client.search_recent_tweets(
    query,
    max_results=n_tweets
)

tweets = [tweet.text for tweet in response.data]

print('\n\n'.join(tweets))

Unauthorized: 401 Unauthorized

In [23]:
%%time
preprocessed_tweets = map(preprocess_tweet, tweets)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


In [24]:
print_tweets(preprocessed_tweets)

0: fantastic meal favorite restaurant foodie
1: watching sunset mountains absolutely breathtaking nature beautiful
2: learning python today challenging rewarding coding programming
3: new book release wait dive weekend books reading
4: cat funniest thing wish caught camera cats pets
5: productive day work feeling accomplished work motivation
6: planning next vacation go suggestions welcome travel wanderlust
7: enjoying quiet evening cup tea simple pleasures relax evening
8: new movie trailer looks amazing definitely going see movies film
9: thinking starting new project ideas beginner project ideas
